# Pandas I exercises 16/10/2026
### Brais Otero Lema

In [1]:
import numpy as np
import pandas as pd

def make_events(n_events, rng, missing=0.):
    """Return a DataFrame of synthetic collision events, one row per particle.

    Each event has 1 + Poisson(4) particles. Columns: event, ptype (category),
    charge, pt (GeV), eta and phi. A fraction `missing` of the eta values is set
    to NaN (failed measurements). `rng` is a numpy Generator: the same seed gives
    the same table.
    """
    n = rng.poisson(4, n_events) + 1                    # particles per event
    event = np.repeat(np.arange(n_events), n)
    size = event.size
    ptype = rng.choice(['pion', 'kaon', 'muon', 'electron'], size, p=[.6, .2, .1, .1])
    df = pd.DataFrame({
        'event': event, 'ptype': pd.Categorical(ptype),
        'charge': rng.choice([-1, 1], size),
        'pt': rng.exponential(10., size),
        'eta': rng.normal(0., 1.5, size),
        'phi': rng.uniform(-np.pi, np.pi, size)})
    if missing > 0:
        df.loc[rng.random(size) < missing, 'eta'] = np.nan
    return df

Exercise 1. Dimuon events.

In [2]:
import numpy as np
import pandas as pd

events = make_events(3_000, np.random.default_rng(2664), missing=.05)

measured_eta = ~np.isnan(events["eta"])
big_pt = events["pt"] > 5
central_eta = np.abs(events["eta"]) < 2.5
mask = measured_eta & big_pt & central_eta
cut_events = events[mask]

dfs = (events, cut_events)
sts = ("Before", "After")

print(f"Before the cut, there are {len(events)} particles in {events["event"].nunique()} events.")
print(f"After the cut, there are {len(cut_events)} particles in {cut_events["event"].nunique()} events.")

muon_events = cut_events[cut_events["ptype"] == "muon"]
opposite_dimuons = len(muon_events.groupby("event").agg(n=("pt", "size"), q=("charge", "sum")).query("n == 2 and q == 0"))

print(f"\nThere are {opposite_dimuons} events with exactly two muons of opposite charge.")

cut_events = cut_events.assign(pt_rel=cut_events["pt"] / cut_events.groupby("event")["pt"].transform("max"))


Before the cut, there are 14982 particles in 3000 events.
After the cut, there are 7846 particles in 2827 events.

There are 43 events with exactly two muons of opposite charge.


Exercise 2. Hit calibration.

In [3]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(2664)
n = 2000
hits = pd.DataFrame({'channel': rng.integers(0, 16, n),
                     'adc': rng.normal(500, 80, n).round()})
hits.loc[rng.random(n) < 0.03, 'channel'] = np.nan    # channel not decoded
gain = pd.Series(rng.normal(0.020, 0.002, 16).round(4), index=range(16))
gain = gain.drop([5, 11])

hits["channel"] = hits["channel"].astype("Int64")
hits_without_channel = sum(hits["channel"].isna())
print(f"There are {hits_without_channel} hits without channel.")

hits = hits.assign(energy=hits["channel"].map(gain))
hits_without_energy = sum(hits["energy"].isna())
because_no_calibration = sum(hits["channel"].isin([5, 11]))
print(f"\nThere are {hits_without_energy} hits with no energy; {hits_without_channel} because they have no channel and {because_no_calibration} because they have no calibration.")

saturation_mask = hits["adc"] > 700
hits.loc[saturation_mask, "energy"] = np.nan
grouped_by_channel = hits.groupby("channel").agg(number=("adc", "size"), mean_energy=("energy", "mean"))
print(f"\nPer channel, number of hits and mean energy:\n{grouped_by_channel}")


There are 57 hits without channel.

There are 264 hits with no energy; 57 because they have no channel and 207 because they have no calibration.

Per channel, number of hits and mean energy:
         number  mean_energy
channel                     
0           121       0.0194
1           111       0.0232
2           124       0.0163
3           129       0.0190
4           136       0.0212
5           101          NaN
6           142       0.0227
7           113       0.0217
8           135       0.0190
9           120       0.0185
10          128       0.0179
11          106          NaN
12          117       0.0217
13          123       0.0184
14          130       0.0208
15          107       0.0211


The channels with no mean energy are number 5 and 11. This is because all of the energies for these two channels are NaN, as can be seen:

In [5]:
for channel in (5, 11):
    print(f"Channel {channel}: {np.all(hits[hits["channel"] == channel]["energy"].isna())}")

Channel 5: True
Channel 11: True
